In [1]:
# NB : This reload your library after each edit, 
# so you don't have to restart the kernel
%load_ext autoreload
%autoreload 2

In [2]:
# --- Imports (replace the TF ones) ---
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image

#from herbs_detection.model import predict, predict_top3, predict_set

In [16]:
PROJECT_ROOT = Path("/home/jouell/code/jouell3/plant_detect")
ALL_IMAGES_DIR = PROJECT_ROOT / "data/raw/all_images"
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS_FROZEN = 10
EPOCHS_FINE = 20

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


# --- Load labels ---
labels = pd.read_csv(PROJECT_ROOT / "data/files_df.csv")
# match images in all_images/ by basename of the original path
labels["basename"] = labels["filename"].apply(lambda f: Path(f).name)
print(labels["name"].value_counts())
print(f"\nTotal: {len(labels)} images")

Using device: cpu
name
sage            437
lemonverbena    342
hyssop          312
lemongrass      294
borage          226
rosemary        205
wintergreen     187
basil           187
lavender        186
mugwort         179
lovage          172
chamomile       167
mint            148
parsley         137
thymee          121
tarragon        114
fennel          110
oregano         109
angelica        104
coriander        84
dill             82
savory           81
chives           75
thyme            53
Name: count, dtype: int64

Total: 4112 images


In [6]:
# Build a dataframe from filenames in all_images/
file_names = sorted([p.name for p in ALL_IMAGES_DIR.iterdir() if p.is_file()])

files_df = pd.DataFrame({"filename": file_names})
files_df["name"] = files_df["filename"].str.split("_", n=1).str[0]

print(f"Found {len(files_df)} files in {ALL_IMAGES_DIR}")
files_df.head()

Found 4112 files in /home/jouell/code/jouell3/plant_detect/data/raw/all_images


,filename,name
0,aneth_0001.jpg,aneth
1,aneth_0003.jpg,aneth
2,aneth_0004.jpg,aneth
3,aneth_0005.jpg,aneth
4,aneth_0007.jpg,aneth


In [17]:
# --- Load images from all_images/ as numpy arrays ---
X, y = [], []
missing = []

for _, row in labels.iterrows():
    img_path = ALL_IMAGES_DIR / row["basename"]
    if not img_path.exists():
        missing.append(row["basename"])
        continue
    img = Image.open(img_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    X.append(np.array(img))
    y.append(row["name"])

if missing:
    print(f"Warning: {len(missing)} images not found in all_images/")

X = np.array(X, dtype=np.float32) / 255.0   # shape: (N, 224, 224, 3), normalized to [0, 1]
y = np.array(y)

le = LabelEncoder()
y_enc = le.fit_transform(y)                  # integer labels
NUM_CLASSES = len(le.classes_)

print(f"\nX shape: {X.shape}")
print(f"y shape: {y_enc.shape}")
print(f"Classes ({NUM_CLASSES}): {le.classes_}")




X shape: (4112, 224, 224, 3)
y shape: (4112,)
Classes (24): ['angelica' 'basil' 'borage' 'chamomile' 'chives' 'coriander' 'dill'
 'fennel' 'hyssop' 'lavender' 'lemongrass' 'lemonverbena' 'lovage' 'mint'
 'mugwort' 'oregano' 'parsley' 'rosemary' 'sage' 'savory' 'tarragon'
 'thyme' 'thymee' 'wintergreen']


In [18]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch

class PlantDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X = X          # (N, 224, 224, 3) float32 in [0,1]
        self.y = y
        self.augment = augment
        self.normalize = transforms.Normalize([0.485, 0.456, 0.406],
                                              [0.229, 0.224, 0.225])
        self.aug = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
        ]) if augment else None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        img = torch.from_numpy(self.X[idx]).permute(2, 0, 1)  # (3, H, W)
        if self.aug:
            img = self.aug(img)
        img = self.normalize(img)
        return img, int(self.y[idx])


In [19]:
# --- Dataset ---

from sklearn.model_selection import train_test_split

idx = np.arange(len(X))
idx_train, idx_test = train_test_split(idx, test_size=0.15, stratify=y_enc, random_state=42)
idx_train, idx_val  = train_test_split(idx_train, test_size=0.15, stratify=y_enc[idx_train], random_state=42)

train_loader = DataLoader(PlantDataset(X[idx_train], y_enc[idx_train], augment=True),
                          batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(PlantDataset(X[idx_val],   y_enc[idx_val]),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(PlantDataset(X[idx_test],  y_enc[idx_test]),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2)



In [20]:
# --- Model: ResNet18 pretrained ---
model = models.resnet18(weights="IMAGENET1K_V1")

# Phase 1: freeze backbone, replace head
for p in model.parameters():
    p.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)  # only head is trainable

model = model.to(DEVICE)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct = 0, 0
    with torch.set_grad_enabled(train):
        for imgs, labs in loader:
            imgs, labs = imgs.to(DEVICE), labs.to(DEVICE)
            out  = model(imgs)
            loss = criterion(out, labs)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * len(imgs)
            correct    += (out.argmax(1) == labs).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

# --- Phase 1: head only ---
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

print("Phase 1 — frozen backbone")
for epoch in range(EPOCHS_FROZEN):
    tl, ta = run_epoch(train_loader, train=True)
    vl, va = run_epoch(val_loader,   train=False)
    print(f"Epoch {epoch+1:02d} | train loss {tl:.3f} acc {ta:.3f} | val loss {vl:.3f} acc {va:.3f}")


Phase 1 — frozen backbone
Epoch 01 | train loss 2.514 acc 0.311 | val loss 1.963 acc 0.450
Epoch 02 | train loss 1.771 acc 0.518 | val loss 1.610 acc 0.537
Epoch 03 | train loss 1.468 acc 0.592 | val loss 1.422 acc 0.600
Epoch 04 | train loss 1.317 acc 0.628 | val loss 1.328 acc 0.613
Epoch 05 | train loss 1.219 acc 0.651 | val loss 1.287 acc 0.613
Epoch 06 | train loss 1.162 acc 0.651 | val loss 1.249 acc 0.625
Epoch 07 | train loss 1.072 acc 0.678 | val loss 1.208 acc 0.644
Epoch 08 | train loss 1.025 acc 0.692 | val loss 1.202 acc 0.630
Epoch 09 | train loss 0.993 acc 0.699 | val loss 1.150 acc 0.648
Epoch 10 | train loss 0.944 acc 0.715 | val loss 1.153 acc 0.646


In [ ]:
# --- Phase 2: unfreeze all layers, fine-tune with low LR ---
for p in model.parameters():
    p.requires_grad = True

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

best_val, patience_cnt, best_state = 0, 0, None
print("\nPhase 2 — fine-tuning")
for epoch in range(EPOCHS_FINE):
    tl, ta = run_epoch(train_loader, train=True)
    vl, va = run_epoch(val_loader,   train=False)
    scheduler.step(vl)
    print(f"Epoch {epoch+1:02d} | train loss {tl:.3f} acc {ta:.3f} | val loss {vl:.3f} acc {va:.3f}")
    if va > best_val:
        best_val = va; best_state = {k: v.clone() for k, v in model.state_dict().items()}; patience_cnt = 0
    else:
        patience_cnt += 1
        if patience_cnt >= 5:
            print("Early stopping"); break

model.load_state_dict(best_state)
print(f"\nBest val accuracy: {best_val:.3f}")



Phase 2 — fine-tuning


In [ ]:
# --- Evaluation ---
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for imgs, labs in test_loader:
        preds = model(imgs.to(DEVICE)).argmax(1).cpu().tolist()
        y_pred.extend(preds)
        y_true.extend(labs.tolist())

print(classification_report(y_true, y_pred, target_names=le.classes_))

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt="d", xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("ResNet18 — Confusion Matrix")
plt.tight_layout(); plt.show()


In [ ]:
import torch
from sklearn.preprocessing import LabelEncoder
import pickle

save_dir = PROJECT_ROOT / "backend/app/models"
save_dir.mkdir(parents=True, exist_ok=True)

# Save model weights (equivalent to model.save_weights() in Keras)
torch.save(model.state_dict(), save_dir / "resnet18_plants.pt")

# Save the LabelEncoder too — you need it to decode predictions back to species names
with open(save_dir / "label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

print("Model saved.")


In [ ]:
# Try it
species, confidence = predict("../data/raw/all_images/thyme_434.jpg")
print(f"Predicted: {species}  ({confidence:.1%} confidence)")

In [ ]:
for species, conf in predict_top3("data/raw/all_images/dill_0.jpg"):
    print(f"  {species:<15} {conf:.1%}")